In [ ]:
from pathlib import Path
import json
import numpy as np
from tqdm import tqdm
from SoccerNet.Downloader import getListGames

SOCCERNET_PATH = Path(r"/path/to/SoccerNet")
FEATURE_NAME = "baidu_soccer_embeddings.npy"
SPLITS = ["train", "valid", "test"]
OUT_FEATURE_FILE = Path("data/features.dat")
OUT_MAPPING_JSON = Path("data/mapping.json")
FRAMERATE = 1
WINDOW_SIZE_SECONDS = 45
PAD_MODE = "edge"

window_size_frame = FRAMERATE * WINDOW_SIZE_SECONDS
l_pad = window_size_frame // 2 + window_size_frame % 2
r_pad = window_size_frame // 2

games = getListGames(SPLITS, task="caption")
if not games:
    raise RuntimeError("No games found.")


def load_half(game, half):
    f = np.load(SOCCERNET_PATH / game / f"{half}_{FEATURE_NAME}")
    return f.reshape(-1, f.shape[-1]).astype(np.float32)


# Pass 1: get total rows + feature dim
sample = load_half(games[0], 1)
feature_dim = sample.shape[-1]
total_rows = 0

for game in tqdm(games, desc="Scan lengths"):
    for half in (1, 2):
        f = load_half(game, half)
        if f.shape[-1] != feature_dim:
            raise ValueError(f"Feature dim mismatch in {game}")
        total_rows += f.shape[0] + l_pad + r_pad

# Pass 2: write memmap
OUT_FEATURE_FILE.parent.mkdir(parents=True, exist_ok=True)
OUT_MAPPING_JSON.parent.mkdir(parents=True, exist_ok=True)

mem = np.memmap(OUT_FEATURE_FILE, mode="w+", dtype=np.float32, shape=(total_rows, feature_dim))
mapping = {}
cursor = 0

for idx, game in enumerate(tqdm(games, desc="Write memmap")):
    entry = {}
    for half in (1, 2):
        padded = np.pad(load_half(game, half), ((l_pad, r_pad), (0, 0)), mode=PAD_MODE)
        mem[cursor : cursor + len(padded)] = padded
        entry[f"half{half}_start"] = cursor
        entry[f"half{half}_len"] = len(padded)
        cursor += len(padded)
    mapping[str(idx)] = entry

mem.flush()
OUT_MAPPING_JSON.write_text(json.dumps(mapping, indent=2))

print(f"Done — {len(games)} games, dim={feature_dim}, "
      f"window={window_size_frame} (pad {l_pad}/{r_pad})\n"
      f"Features: {OUT_FEATURE_FILE}\nMapping:  {OUT_MAPPING_JSON}")